# TabPFN-3 auf Colab

Rechnet die Fold-Schleife für TabPFN-3 auf der GPU und legt `predictions.parquet` ab.
Das Modell bleibt hier, nur der Prognose-Frame wandert zurück.

**Vor dem Start:** Laufzeit → Laufzeittyp ändern → T4 GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 — Repository und Abhängigkeiten

In [ ]:
!git clone --depth 1 https://github.com/mflindt/pv-forecast.git
%cd pv-forecast
!pip install -q -e ".[gpu]"

## 2 — Modell-Input

`data/processed/` liegt nicht im Repository und wird hier aus den Rohdaten neu gebaut.

In [ ]:
!python -m pvforecast.preprocessing

## 3 — Probelauf

Erst ein Fold. Bricht das ab, ist der Kontext zu groß für die Karte. Dann in
`configs/tabpfn.yaml` entweder `contexts` auf eine kleinere Größe setzen oder
`daylight_training: true` — das halbiert die Zeilen auf die Tagstunden.

In [ ]:
!python main.py --config configs/tabpfn.yaml --stage train --folds 1

## 4 — Vollständiger Lauf

In [ ]:
!python main.py --config configs/tabpfn.yaml --stage train

## 5 — Ergebnis herunterladen

Lokal danach:

    python main.py --stage evaluate --runs <lokaler_lauf> <colab_lauf>

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

runs = [p for p in Path("reports").iterdir() if p.is_dir() and p.name != "latest"]
run = sorted(runs)[-1]
print(f"{run.name}: {sorted(p.name for p in run.iterdir())}")
files.download(shutil.make_archive(run.name, "zip", run))